In [1]:
!pip install pandas numpy scikit-learn matplotlib seaborn

In [3]:
import pandas as pd

df = pd.read_csv("transactions_completed.csv")
df.head()

,order_id,customer_id,order_date,region,segment,payment_method,channel,status,product_category,product_name,unit_price,quantity,discount_pct,gross_sales,discount_amount,net_sales,year_month
0,1,2126,2024-10-24,West,Budget,Card,Online,completed,Clothing,Jeans,29,3,0.05,87,4.35,82.65,2024-10
1,2,2459,2024-11-12,South,Budget,Card,Online,completed,Sports,Dumbbells,15,1,0.05,15,0.75,14.25,2024-11
2,3,1860,2024-02-27,South,Budget,Apple Pay,Online,completed,Sports,Dumbbells,80,3,0.15,240,36.00,204.00,2024-02
3,4,2294,2024-02-03,South,Budget,Card,Online,completed,Electronics,Headphones,199,2,0.05,398,19.90,378.10,2024-02
4,5,2130,2024-09-19,South,Mainstream,Card,Online,completed,Home,Chair,49,3,0.10,147,14.70,132.30,2024-09


In [5]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("customer_sales_analytics.db")

# Assuming 'df' DataFrame is already loaded from transactions_completed.csv
# Write the DataFrame to the SQL database, replacing if the table already exists
df.to_sql('transactions_completed', conn, if_exists='replace', index=False)

df = pd.read_sql("SELECT * FROM transactions_completed", conn)
df.head()

conn.close()

In [6]:
df.info()
df.isnull().sum()

df['order_date'] = pd.to_datetime(df['order_date'])
df = df.drop_duplicates()

print("Final shape:", df.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10772 entries, 0 to 10771
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          10772 non-null  int64  
 1   customer_id       10772 non-null  int64  
 2   order_date        10772 non-null  object 
 3   region            10772 non-null  object 
 4   segment           10772 non-null  object 
 5   payment_method    10772 non-null  object 
 6   channel           10772 non-null  object 
 7   status            10772 non-null  object 
 8   product_category  10772 non-null  object 
 9   product_name      10772 non-null  object 
 10  unit_price        10772 non-null  int64  
 11  quantity          10772 non-null  int64  
 12  discount_pct      10772 non-null  float64
 13  gross_sales       10772 non-null  int64  
 14  discount_amount   10772 non-null  float64
 15  net_sales         10772 non-null  float64
 16  year_month        10772 non-null  object

In [7]:
from datetime import datetime

snapshot_date = df['order_date'].max()

rfm = df.groupby('customer_id').agg({
    'order_date': lambda x: (snapshot_date - x.max()).days,
    'order_id': 'count',
    'net_sales': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm.head()

,Recency,Frequency,Monetary
customer_id,,,
1000,33,13,4559.50
1001,30,5,636.00
1002,32,6,1395.75
1003,2,4,911.10
1004,284,4,2402.05


In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

kmeans = KMeans(n_clusters=3, random_state=42)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

rfm.head()

,Recency,Frequency,Monetary,Cluster
customer_id,,,,
1000,33,13,4559.50,1
1001,30,5,636.00,2
1002,32,6,1395.75,2
1003,2,4,911.10,2
1004,284,4,2402.05,0


In [9]:
from sklearn.metrics import silhouette_score

score = silhouette_score(rfm_scaled, rfm['Cluster'])
print("Silhouette Score:", score)

Silhouette Score: 0.31608822920192214


In [10]:
rfm.to_csv("customer_segments.csv")